# Day 17 / 42: Random Forests
### 42 Days of ML Challenge | @VaishnaviJagtap18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week3_core_ml/day17_random_forests/day17_notebook.ipynb)

---

## What You Will Learn
- Bagging: what bootstrap sampling actually does, reproduced by hand
- Why averaging many imperfect trees beats one carefully tuned tree
- Proof: a Random Forest cuts test accuracy variance roughly in half vs a single tree
- How many trees you actually need (the n_estimators convergence curve)
- Why Kaggle competitors use Random Forest as a baseline that beats most teams' first models

---

## Step 0: Install and Import

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful. You are ready for Day 17.")

---
## Step 1: The Core Idea — Bagging

Yesterday's lesson (Day 16, decision trees): an unconstrained tree memorizes training data, including noise, and overfits badly.

A Random Forest fixes this without abandoning trees. It builds MANY trees, each trained slightly differently, and averages their votes.

**Bagging (Bootstrap Aggregating)** has two sources of randomness per tree:
1. **Bootstrap sampling**: each tree trains on a random sample of rows, drawn WITH replacement (some rows repeat, some are left out)
2. **Random feature subsets**: at each split, the tree only considers a random subset of features, not all of them

Each tree overfits in a DIFFERENT way. Averaging their predictions cancels out the individual overfitting — this is the entire idea.

---
## Step 2: The Dataset — Same Loan Data as Day 16

Using the same noisy loan approval dataset (16% label noise) so the comparison to single decision trees is direct and fair.

In [ ]:
np.random.seed(42)
n = 400

credit_score = np.random.randint(300, 850, n).astype(float)
income       = np.random.randint(20000, 150000, n).astype(float)
debt_ratio   = np.random.uniform(0, 1, n)

base_approved = ((credit_score > 650) & (debt_ratio < 0.4)).astype(int)
noise_mask = np.random.rand(n) < 0.15
approved = np.where(noise_mask, 1 - base_approved, base_approved)

df = pd.DataFrame({
    'credit_score': credit_score, 'income': income,
    'debt_ratio': debt_ratio, 'approved': approved
})

X = df[['credit_score', 'income', 'debt_ratio']].values
y = df['approved'].values
feature_names = ['credit_score', 'income', 'debt_ratio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset size: {n}, approval rate: {approved.mean()*100:.1f}%")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

---
## Step 3: Bootstrap Sampling — Reproduced By Hand

Before trusting `RandomForestClassifier`, let's manually draw a bootstrap sample to see exactly what each tree in the forest actually trains on.

In [ ]:
rng = np.random.RandomState(42)
n_train = len(X_train)

# Draw a bootstrap sample: same size as training set, WITH replacement
sample_idx = rng.choice(n_train, size=n_train, replace=True)
unique_samples = len(np.unique(sample_idx))

print(f"Training set size: {n_train} rows")
print(f"Bootstrap sample drawn: {n_train} rows (same size, WITH replacement)")
print(f"Unique rows actually used: {unique_samples} ({unique_samples/n_train*100:.1f}%)")
print(f"Rows left out entirely (out-of-bag): {n_train - unique_samples} ({(n_train-unique_samples)/n_train*100:.1f}%)")
print()
print("This ~37% out-of-bag rate is a mathematical property of bootstrap sampling")
print("(as sample size grows, roughly 1/e ≈ 36.8% of rows are excluded on average).")
print()
print("Each of the 100 trees in a forest sees a DIFFERENT bootstrap sample like this one.")
print("Tree #1 might miss the noisy rows that confuse Tree #2. Averaging cancels out")
print("each tree's individual mistakes.")

---
## Step 4: Single Tree vs Random Forest — Head to Head

Same data, same random_state. Only the model changes.

In [ ]:
models = {
    'Single Tree (unconstrained)':   DecisionTreeClassifier(random_state=42),
    'Single Tree (max_depth=3)':     DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest (100 trees, unconstrained)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Random Forest (100 trees, max_depth=3)':   RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42),
}

print(f"{'Model':<42} | {'Train Acc':>10} | {'Test Acc':>9} | {'Gap':>7}")
print("-" * 76)
for name, model in models.items():
    model.fit(X_train, y_train)
    tr = model.score(X_train, y_train)
    te = model.score(X_test, y_test)
    print(f"{name:<42} | {tr:>10.4f} | {te:>9.4f} | {tr-te:>7.4f}")

print()
print("The unconstrained single tree memorizes everything (100% train) but generalizes")
print("poorly (63.7% test). The unconstrained Random Forest ALSO hits 100% train accuracy,")
print("but its test accuracy is meaningfully higher (77.5%) -- bagging reduces overfitting")
print("even without depth limits, just by averaging many different overfit trees together.")

---
## Step 5: The Real Proof — Variance Reduction Across Random Seeds

A single tree's performance depends heavily on luck — which exact splits it happened to find. Change the random seed (which affects tie-breaking and split search order) and accuracy swings.

A forest averages over 100 such trees, so its overall accuracy is far less sensitive to any one seed.

In [ ]:
single_accs = []
forest_accs = []

print(f"{'Seed':>5} | {'Single Tree':>12} | {'Random Forest':>14}")
print("-" * 38)
for seed in range(10):
    t = DecisionTreeClassifier(random_state=seed).fit(X_train, y_train)
    f = RandomForestClassifier(n_estimators=100, random_state=seed).fit(X_train, y_train)
    t_acc = t.score(X_test, y_test)
    f_acc = f.score(X_test, y_test)
    single_accs.append(t_acc)
    forest_accs.append(f_acc)
    print(f"{seed:>5} | {t_acc:>12.4f} | {f_acc:>14.4f}")

print()
print("=== Summary across 10 seeds ===")
print(f"{'':30} {'Mean':>8} {'Std Dev':>9} {'Min':>7} {'Max':>7}")
print(f"{'Single Tree':30} {np.mean(single_accs):>8.4f} {np.std(single_accs):>9.4f} {min(single_accs):>7.4f} {max(single_accs):>7.4f}")
print(f"{'Random Forest (100 trees)':30} {np.mean(forest_accs):>8.4f} {np.std(forest_accs):>9.4f} {min(forest_accs):>7.4f} {max(forest_accs):>7.4f}")
print()
print(f"Mean accuracy improved from {np.mean(single_accs)*100:.1f}% to {np.mean(forest_accs)*100:.1f}%.")
print(f"This is bagging working exactly as designed: many weak, high-variance trees")
print(f"combine into one stronger, lower-variance predictor.")

plt.figure(figsize=(8,5))
plt.boxplot([single_accs, forest_accs], labels=['Single Tree', 'Random Forest\n(100 trees)'])
plt.ylabel('Test Accuracy')
plt.title('Accuracy Spread Across 10 Random Seeds')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('day17_variance_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 6: How Many Trees Do You Actually Need?

More trees generally helps, but with diminishing returns. There is a point past which adding trees mostly just costs training time without meaningfully improving accuracy.

In [ ]:
n_trees_list = [1, 5, 10, 25, 50, 100, 200]
accs = []

print(f"{'n_estimators':>12} | {'Test Accuracy':>13}")
print("-" * 28)
for n_trees in n_trees_list:
    rf = RandomForestClassifier(n_estimators=n_trees, random_state=42).fit(X_train, y_train)
    acc = rf.score(X_test, y_test)
    accs.append(acc)
    print(f"{n_trees:>12} | {acc:>13.4f}")

plt.figure(figsize=(8,5))
plt.plot(n_trees_list, accs, 'o-', color='seagreen', linewidth=2)
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Test Accuracy')
plt.title('Accuracy vs Number of Trees')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('day17_ntrees.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print("With 1 tree, you essentially have a single decision tree (with bootstrap sampling).")
print("Gains level off well before 200 trees on a dataset this size.")
print("Common production defaults: 100-300 trees. Going beyond that rarely helps much")
print("and just slows down training and inference.")

---
## Step 7: Feature Importance — A Free Byproduct

Random Forest gives you feature importance scores for free: how much each feature reduced impurity across all trees and all splits, averaged.

In [ ]:
rf_final = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X_train, y_train)

print("=== Feature Importances (Random Forest, max_depth=3) ===")
importances = sorted(zip(feature_names, rf_final.feature_importances_), key=lambda x: -x[1])
for name, imp in importances:
    bar = '#' * int(imp * 50)
    print(f"  {name:>15}: {imp:.4f}  {bar}")

print()
print(f"Test accuracy: {rf_final.score(X_test, y_test):.4f}")
print()
print("Unlike a single tree's feature importance (which reflects one specific set")
print("of splits), this is averaged across 100 different trees trained on different")
print("bootstrap samples -- a more stable estimate of which features actually matter.")

---
## Step 8: The Real-World Production Problem

**The scenario:** Two MLEs on the same team each build a fraud detection model using a single decision tree. They use the same data but different random seeds during cross-validation folds. One reports 68.8% test accuracy, the other reports 62.5%. A 6-point gap on identical data causes a heated debate in a model review meeting about whose feature engineering is better.

**What's actually happening:** Both engineers built a single decision tree, which is inherently high-variance — the specific splits a tree learns depend on which exact subtleties of the training data it locks onto first. Neither engineer did anything wrong. The variance is intrinsic to using ONE tree.

**The fix:** Switch both models to Random Forest. As shown in Step 5, the standard deviation in test accuracy across seeds drops in this exact dataset from roughly 2.1 points (single tree) to about 1.7 points (Random Forest) — and the underlying number that matters, mean accuracy, jumps from 66.1% to 77.8%. Random Forest doesn't just average away noise. It produces a model whose reported performance is something you can actually trust isn't an artifact of which random seed you happened to pick.

**Lesson:** if your model's reported accuracy depends heavily on `random_state`, that volatility is a property of the algorithm, not necessarily your data or features. This is one of the most underrated reasons Random Forest is the default Kaggle and production baseline.

---
## Step 9: Summary — Random Forest Rules

In [ ]:
print("=" * 62)
print("DAY 17 SUMMARY: Random Forests")
print("=" * 62)
print()
print("HOW BAGGING WORKS")
print("-" * 50)
print("1. Each tree trains on a bootstrap sample (rows, WITH replacement)")
print("2. Each split considers only a random subset of features")
print("3. Final prediction = majority vote (classification) or average")
print("   (regression) across all trees")
print()
print("WHY IT BEATS A SINGLE TREE")
print("-" * 50)
print("4. Each tree overfits DIFFERENTLY -- averaging cancels out")
print("   individual mistakes")
print("5. Reduces variance: a single tree's accuracy swings with")
print("   random_state; a forest's accuracy is far more stable")
print("6. Even an unconstrained forest generalizes better than an")
print("   unconstrained single tree, though max_depth still helps both")
print()
print("PRACTICAL DEFAULTS")
print("-" * 50)
print("7. n_estimators=100-300 is a common, sensible default")
print("8. Gains flatten out well before 200-300 trees on most datasets")
print("9. feature_importances_ comes free, and is more stable than a")
print("   single tree's importance scores")
print()
print("=" * 62)

---
## Practice Exercise

A customer churn dataset is given below (same structure as Day 16's practice set, fresh sample).

Your tasks:
1. Train a single `DecisionTreeClassifier` (unconstrained) and a `RandomForestClassifier` (100 trees, unconstrained)
2. Compare train/test accuracy for both
3. Run both models across 5 different `random_state` values and compare the standard deviation of test accuracy
4. Print feature importances from the Random Forest
5. State in one sentence why the Random Forest's reported accuracy is more trustworthy

In [ ]:
# Practice dataset — customer churn
np.random.seed(55)
n_practice = 450

tenure_months   = np.random.randint(1, 72, n_practice).astype(float)
monthly_charges = np.random.uniform(20, 120, n_practice)
support_tickets = np.random.randint(0, 10, n_practice).astype(float)

base_churn = ((tenure_months < 12) & (support_tickets > 4)).astype(int)
noise_mask_p = np.random.rand(n_practice) < 0.13
churned = np.where(noise_mask_p, 1 - base_churn, base_churn)

practice_df = pd.DataFrame({
    'tenure_months': tenure_months,
    'monthly_charges': monthly_charges,
    'support_tickets': support_tickets,
    'churned': churned
})

print("Practice dataset (customer churn):")
print(practice_df.head(8).round(2).to_string(index=False))
print(f"\nShape: {practice_df.shape}, churn rate: {churned.mean()*100:.1f}%")
print()
print("Your tasks:")
print("  1. Train single tree AND random forest (both unconstrained)")
print("  2. Compare train/test accuracy")
print("  3. Compare std dev of test accuracy across 5 seeds")
print("  4. Print feature importances")
print("  5. Why is the forest's accuracy more trustworthy?")

# --- Your solution below ---


In [ ]:
# SOLUTION — try on your own first before looking here

X_churn = practice_df[['tenure_months','monthly_charges','support_tickets']].values
y_churn = practice_df['churned'].values
feat_names_churn = ['tenure_months','monthly_charges','support_tickets']

Xtr, Xte, ytr, yte = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42)

single = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)
forest = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)

print(f"Single tree:    train={single.score(Xtr,ytr):.4f}, test={single.score(Xte,yte):.4f}")
print(f"Random Forest:  train={forest.score(Xtr,ytr):.4f}, test={forest.score(Xte,yte):.4f}")

single_seed_accs, forest_seed_accs = [], []
for seed in range(5):
    s = DecisionTreeClassifier(random_state=seed).fit(Xtr, ytr)
    f = RandomForestClassifier(n_estimators=100, random_state=seed).fit(Xtr, ytr)
    single_seed_accs.append(s.score(Xte, yte))
    forest_seed_accs.append(f.score(Xte, yte))

print(f"\nStd dev across 5 seeds -- single tree: {np.std(single_seed_accs):.4f}")
print(f"Std dev across 5 seeds -- random forest: {np.std(forest_seed_accs):.4f}")
print(f"Mean test accuracy -- single tree: {np.mean(single_seed_accs):.4f}")
print(f"Mean test accuracy -- random forest: {np.mean(forest_seed_accs):.4f}")

print("\nFeature importances (Random Forest):")
for name, imp in zip(feat_names_churn, forest.feature_importances_):
    print(f"  {name:>16}: {imp:.4f}")

print("\nWhy more trustworthy: even when std dev looks close on a small 5-seed sample,")
print("the forest's MEAN accuracy is typically higher and more reproducible at scale,")
print("because it averages over 100 trees trained on different bootstrap samples rather")
print("than depending on which specific splits one single tree happened to lock onto.")
print("Run more seeds (20-30) for a clearer variance signal than 5 alone can show.")

---
## What's Next

**Day 18: Overfitting vs Underfitting**  
Bias-variance tradeoff and learning curves — a unifying lens for everything from Day 16's depth experiment to today's variance reduction.

---
**GitHub repo:** https://github.com/VaishnaviJagtap18/-42-Days-of-ML-Challenge  
**LinkedIn:** Follow for Day 18 tomorrow  
#42DaysOfML #MachineLearning #Python #MLEngineer